In [ ]:
import os
import glob
import numpy as np
import cv2

def load_image(path: str):
    ext = os.path.splitext(path)[1].lower()

    img = None
    if ext in [".tif", ".tiff"]:
        try:
            import tifffile
            img = tifffile.imread(path)
        except Exception:
            img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    else:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if img.ndim > 2:
        img = np.squeeze(img)
        if img.ndim != 2:
            img = img[..., 0]

    return img.astype(np.float32)

def subtract_plane(img: np.ndarray):
    img = img.astype(np.float32)
    h, w = img.shape
    Y, X = np.mgrid[0:h, 0:w]

    A = np.column_stack((X.ravel(), Y.ravel(), np.ones(h * w, dtype=np.float32)))
    Z = img.ravel()

    coeffs, *_ = np.linalg.lstsq(A, Z, rcond=None)
    a, b, c = coeffs

    plane = (a * X + b * Y + c).astype(np.float32)
    return (img - plane).astype(np.float32)

def normalize_percentile(img: np.ndarray, p_low: float = 1.0, p_high: float = 99.0):
    img = img.astype(np.float32)
    lo, hi = np.percentile(img, (p_low, p_high))
    img = np.clip(img, lo, hi)
    denom = img.max() - img.min()
    if denom < 1e-12:
        return np.zeros_like(img, dtype=np.float32)
    return ((img - img.min()) / denom).astype(np.float32)

def resize_image(img: np.ndarray, size=(256, 256)):
    return cv2.resize(img.astype(np.float32), size, interpolation=cv2.INTER_AREA).astype(np.float32)

def save_png16(img01: np.ndarray, out_path: str):
    img01 = np.clip(img01.astype(np.float32), 0.0, 1.0)
    img_u16 = (img01 * 65535.0).round().astype(np.uint16)
    cv2.imwrite(out_path, img_u16)

In [ ]:
input_folder =  r"path/to/raw_images"
output_folder = r"path/to/preprocessed_images"
os.makedirs(output_folder, exist_ok=True)

patterns = ["*.tif", "*.tiff", "*.png"]
image_paths = []
for pat in patterns:
    image_paths.extend(glob.glob(os.path.join(input_folder, pat)))

for path in image_paths:
    img = load_image(path)            
    img = subtract_plane(img)               
    img = normalize_percentile(img)   
    img = resize_image(img, (256, 256))

    base = os.path.splitext(os.path.basename(path))[0]
    out_path_img = os.path.join(output_folder, base + ".png")
    save_png16(img, out_path_img)

print(f"Processed {len(image_paths)} images. Output saved in: {output_folder}")